# Instruction Tuning

In this notebook, we will fine-tune a pretrained language model to carry out two
instructions: **copy a headline** and **extract the country named in it**.
We will inspect the data, train the model, save it, and compare its answers with
those from the starting checkpoint.

The tasks are small enough to try on a CPU or Apple Silicon laptop. More importantly,
they let us ask a clear question: **does changing the instruction change the answer
appropriately, even when the input stays the same?**


## What instruction SFT means

Instruction SFT is supervised fine-tuning in which a natural-language instruction is part of the input and the desired response is the target. Each row has an instruction, an input, and an output, so the model learns a small contract such as “copy this text” or “return its country.”

The model remains one causal language model with one LM head; there is no separate copy head or country classifier. Instruction SFT is not RLHF.

The distinction matters because the model is still learning next-token probabilities. We are not giving it a special rule engine. During training it reads the instruction and input as context, then learns to continue with the supplied response. The clean task contrast lets us see whether the continuation changes for the right reason.

## The two tasks

Every source input appears twice. The first row says **copy the input exactly**. Its response must preserve every character that matters for the task, including case, punctuation, spacing inside the text, and word order. The second row says **extract only the country name**. Its response is the explicitly identified country, not the whole sentence.

For example, one pair can look like this:

```text
Instruction: Copy the headline exactly, including punctuation.
Input:      Japan: students plan a science fair
Response:   Japan: students plan a science fair

Instruction: Extract the one explicitly named country from the headline.
Input:      Japan: students plan a science fair
Response:   Japan
```

The input is identical in the pair, while the outputs are deliberately different. A model that emits the whole input for both rows has not followed the task switch, even if its text looks fluent. These are small, procedural labels designed to make the behavior easy to inspect; they are not claims about news or broad language ability.

## Set up the environment

Use Python **3.12.5** in `pyenv genai3` and the matching notebook kernel.
Install the accompanying requirements before running the notebook:

```bash
PYENV_VERSION=genai3 python -m pip install -r requirements.txt
PYENV_VERSION=genai3 python -m ipykernel install --user --name genai3 --display-name "Python (genai3)"
```

The tested package pins are PyTorch **2.14.1**, Transformers **5.18.0**,
datasets **5.0.1**, Accelerate **1.15.0**, Hugging Face Hub **1.33.0**,
tokenizers **0.23.2**, pandas **3.0.6**, SacreBLEU **2.6.0**, and psutil **7.2.2**.
Kernel/execution dependencies are pinned in `requirements.txt` too.

Open the notebook from its own directory. The first model download needs network;
no Hub login is required. The helpers in `sft_utils.py` handle padding, monitoring,
and evaluation mechanics, while the model and SFT training steps remain visible below.


## Choose a profile

Make the profile choice at the top of the notebook, before loading the trainer. `short` reads the prepared 64-group training selection: 128 rows, one epoch, 32 updates with batch size 4. `full` reads all 140 prepared training groups: 280 rows, three epochs, 210 updates with batch size 4.

The validation and test portions do not change when you switch profiles. Each contains 30 complete groups, or 60 rows. This makes the final comparison fair: a profile changes the amount of learning, not the held-out prompts.

The cells report losses, update counts, and run details as they execute. Treat those values as the output of your selected run rather than assuming that `full` must solve every row. A short run can underfit, while a lower loss can occur without a matching increase in exact task success.
| Profile | Train headlines / examples | Validation | Test | Epochs |
|---|---:|---:|---:|---:|
| `short` | 64 / 128 | 30 / 60 | 30 / 60 | 1 |
| `full` | 140 / 280 | 30 / 60 | 30 / 60 | 3 |

Select one profile in the next cell. No dataset-building or splitting is performed
during the run; it simply chooses the prepared files in `data/<profile>/`.


In [ ]:
PROFILE = "full"  # Choose "short" or "full".
# Uncomment and execute below command to generate the dataset first (if not already present)
#!python prepare_datasets.py

In [2]:
import sft_utils as utils
import torch
from pathlib import Path
from IPython.display import display
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, set_seed

set_seed(42)
DEVICE = utils.select_device()
settings = utils.PROFILES[PROFILE]
OUTPUT_DIR = Path("artifacts") / PROFILE

print("Profile:", PROFILE)
print("Device:", DEVICE)
print("Epochs:", settings["epochs"])
print("Model:", utils.MODEL_NAME)


Profile: full
Device: mps
Epochs: 3
Model: gpt2


## Load the prepared JSON data

The notebook reads the supplied preprepared JSON split files for training, validation, and test. Load the files directly and keep each record's fields available:

- `group_id` identifies the shared input group.
- `task` distinguishes `exact_copy` from `extract_country`.
- `instruction`, `input`, and `output` are the text used to form the example.
- the country label and group/task IDs are useful for checking the data, but are not placed in the prompt.

There is no split-building step in this notebook. Do not generate new splits or derive labels from the text; use the prepared records and their supplied profile selection. Check the row counts and print one paired example so the visible instruction, input, and reference output are clear before tokenization.

The complete source grid contains 200 groups and 400 rows (the selected short profile contains 124 groups and 248 rows): two rows per group. It covers 20 countries through 10 procedural templates. The templates place the country at different positions, which gives the extraction task a simple but meaningful pattern to learn. The outputs are deterministic teaching labels, so the reference is known exactly for scoring.

In [3]:
data, groups = utils.load_profile(PROFILE)
display(utils.data_summary(data, groups))
display(utils.show_examples(data["train"]))


,Headlines,Examples,Copy,Extract
Split,,,,
train,140,280,140,140
validation,30,60,30,30
test,30,60,30,30


,Instruction,Input,Expected response
0,"Copy the headline exactly, including punctuation.",Argentina: students plan a science fair,Argentina: students plan a science fair
1,Extract the one explicitly named country from ...,Argentina: students plan a science fair,Argentina
2,Repeat the input headline exactly as written.,Argentina: library starts a reading club,Argentina: library starts a reading club
3,Name the single country explicitly written in ...,Argentina: library starts a reading club,Argentina


## Keep paired rows together

A source group consists of one copy row and one country-extraction row with the same input. Both rows must be in the same prepared split. Otherwise, training could show the exact input in one task and testing could ask for the other task; an apparently good extraction result could then be memorization rather than instruction following.

The full profile contains 140 training groups; the short profile contains 64. Both use 30 validation groups and 30 test groups. The loading utility checks the counts and confirms that group IDs do not cross partitions. The two rows in every group should also be present exactly once in their partition.

Every partition contains the same 20 country names. Validation and test use held-out headline forms from this small procedural template family. This is a controlled test with familiar countries, not evidence of broad real-world generalization. Use validation loss to monitor the run and, for the full profile, to choose the checkpoint. Keep the test rows untouched until the final comparison.

## Start with GPT-2-small

Load the pretrained `gpt2` checkpoint, which is GPT-2-small with about 124 million parameters, together with its matching tokenizer. Use a causal language-model class, not a sequence-classification class. If the tokenizer has no padding token, the notebook assigns a safe padding token and keeps its padded labels ignored.

The starting model is useful because it already knows ordinary English patterns; it is not a randomly initialized demonstration model. It has not, however, been trained on this exact instruction-and-response format. The notebook utilities handle tokenization, padding, and the small memory telemetry report, so you can concentrate on the data contract and the results rather than building model plumbing.

For the final comparison, reload the untouched starting checkpoint after training. The utility loads one model at a time to limit memory use. The baseline and tuned models must use the same tokenizer and prompt renderer.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    utils.MODEL_NAME, revision=utils.MODEL_REVISION, clean_up_tokenization_spaces=False
)
tokenizer.add_special_tokens({"pad_token": "<pad>"})

model = AutoModelForCausalLM.from_pretrained(
    utils.MODEL_NAME, revision=utils.MODEL_REVISION,
    dtype=torch.float32, attn_implementation="eager"
)
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

print(f"Trainable parameters: {model.num_parameters():,}")



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 32968.13it/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Trainable parameters: 124,440,576


## Train on the response, not the prompt

Render each row as a prompt followed by its response, then append the tokenizer's EOS token:

```text
###Instruction:
Extract the one explicitly named country from the headline.
###Input:
Japan: students plan a science fair
###Response:Japan<EOS>
```

Tokenize the prompt and response separately, concatenate them, and build labels for the causal-language-model loss. A simple tokenizer-dependent diagram is:

```text
input_ids: [P0 P1 P2 P3 P4 A0 A1 EOS PAD PAD]
labels:    [-100 -100 -100 -100 -100 A0 A1 EOS -100 -100]
attention: [  1   1   1   1   1  1  1   1   0   0]
             ignored prompt       supervised answer   ignored padding
```

The letters are positions, not guaranteed words: GPT-2 may split one visible word into several tokens. `-100` makes cross-entropy ignore prompt and padding positions. The model still reads the prompt as context, but only response tokens and EOS teach the target behavior. EOS teaches generation where the response can stop, and padding must not contribute to the loss. The first encoded example is inspected by the notebook so this boundary is concrete rather than hidden in a utility.
A distinct `<pad>` token is added because GPT-2 has no default padding token.
Real EOS tokens still receive supervision; only added padding is ignored.
The following output shows the actual prompt/response boundary for one example.


In [5]:
train_dataset = utils.prepare_training_data(data["train"], tokenizer)
validation_dataset = utils.prepare_training_data(data["validation"], tokenizer)
collator = utils.ResponseOnlyCollator(tokenizer)

display(utils.show_response_boundary(data["train"][1], tokenizer))


,Part,Text,Labels,Token count
0,Prompt (read as context),###Instruction:\nExtract the one explicitly na...,-100 (ignored),34
1,Response + EOS (predicted),Argentina<|endoftext|>,Actual response/EOS IDs,4


## Fine-tune with `Trainer`

The prepared encodings go to Hugging Face `Trainer` with the causal language model, response-only labels, batch size 4, and the epoch/update setting selected by `PROFILE`. `Trainer` performs the minibatch loop and AdamW updates; the model predicts the next token at each supervised response position. It logs training and validation loss while the run proceeds.

For `short`, the 128 rows make 32 batches in one epoch. For `full`, 280 rows make 70 batches per epoch, so three epochs produce 210 optimizer updates. The displayed loss is a next-token training signal, not accuracy. It can fall when token probabilities improve even if an entire copied sentence still has one wrong comma or the country response is wrong.

The notebook utilities also manage padding, generation, scoring, and memory telemetry. You do not need to write a custom PyTorch training loop for this lesson. Read the trainer log and the validation values, then let the selected profile finish before evaluating the test rows.
We evaluate on **validation**, not test, during training. For the full profile,
validation loss selects the checkpoint; for the short one-epoch profile we keep its
final checkpoint. Saving one epoch checkpoint enables recovery if a run is interrupted.
The helper's memory samples are diagnostics, not guaranteed peak-memory measurements.


In [6]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=settings["epochs"],
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=(PROFILE == "full"),
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    use_cpu=(DEVICE == "cpu"),
    optim="adamw_torch",
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    report_to="none",
    logging_steps=10,
    seed=42,
    disable_tqdm=True,
)

monitor = utils.RunMonitor(DEVICE)
trainer = Trainer(
    model=model,
    args=training_args,
    processing_class=tokenizer,
    data_collator=collator,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    callbacks=[monitor],
)
assert trainer.args.device.type == DEVICE

train_result = trainer.train()
validation_result = trainer.evaluate()
print("Training loss:", train_result.training_loss)
print("Validation loss:", validation_result["eval_loss"])


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50257}.


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


{'loss': '1.121', 'grad_norm': '24.62', 'learning_rate': '4.786e-05', 'epoch': '0.1429'}


{'loss': '0.2374', 'grad_norm': '3.883', 'learning_rate': '4.548e-05', 'epoch': '0.2857'}


{'loss': '0.1503', 'grad_norm': '18.8', 'learning_rate': '4.31e-05', 'epoch': '0.4286'}


...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  9.07it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  9.03it/s]

{'train_runtime': '74.55', 'train_samples_per_second': '11.27', 'train_steps_per_second': '2.817', 'train_loss': '0.0826', 'epoch': '3'}


[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


{'eval_loss': '0.0004102', 'eval_runtime': '0.295', 'eval_samples_per_second': '203.4', 'eval_steps_per_second': '50.85', 'epoch': '3'}
Training loss: 0.08259548334227432
Validation loss: 0.0004102314414922148


## Save and reload the selected model

Save the model weights, tokenizer, and run configuration to a local output directory after training. Reload both from that directory with the same causal-language-model and tokenizer classes, then evaluate the reloaded artifact through the comparison helper.

For `full`, choose the checkpoint with the best validation loss; never use test results to choose it. For `short`, use the final checkpoint from the selected one-epoch run. The test set is only for the final baseline-versus-tuned comparison, after this choice has been made.

Reloading is part of the lesson: the object used for scoring should be the saved model that a reader could use later, not merely the in-memory object left at the end of a cell.
The comparison helper reloads the saved tokenizer and model locally. It then loads
the untouched starting checkpoint separately. We release the trainer first so optimizer state and
multiple models do not need to remain on the accelerator during evaluation.


In [7]:
trainer.model.config.use_cache = True
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
run = utils.save_run_summary(
    OUTPUT_DIR, PROFILE, DEVICE, trainer,
    train_result, validation_result, monitor
)

print("Saved model:", OUTPUT_DIR)
print("Optimizer updates:", run["updates"])
del trainer, model
utils.clear_memory()



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.26it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.23it/s]

Saved model: artifacts/full
Optimizer updates: 210


## Compare baseline and tuned generation

Build the final evaluation list from all 30 test groups, including both rows for every group. Feed the untouched starting GPT-2-small and the reloaded tuned model the **same prompts in the same order**. Keep the tokenizer, prompt format, device choice, and decoding settings identical. Use greedy generation with `do_sample=False` and `max_new_tokens=64`. Decode only the continuation after the response marker; do not score the prompt itself.

The output table should retain the group ID, input, instruction, reference, baseline response, and tuned response. A few rows make errors visible, while the aggregate metrics summarize all 60 test rows. Do not replace the paired test list with a convenient sample. The baseline is the untouched model's answer to those prompts, and the tuned answer is produced after the chosen checkpoint is reloaded. The notebook displays the actual generations from the run; it does not fill in expected performance in advance.

In [8]:
baseline_predictions, tuned_predictions = utils.compare_saved_model(
    OUTPUT_DIR, data["test"], DEVICE
)
display(utils.comparison_table(baseline_predictions, tuned_predictions))



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 9362.43it/s]


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 33222.21it/s]

,Group,Input,Instruction,Expected,Starting model,Tuned model
0,group-argentina-10,Volunteers plant new trees in Argentina,Repeat the input headline exactly as written.,Volunteers plant new trees in Argentina,\nThe project is now complete.\n###Contact:\nT...,Volunteers plant new trees in Argentina
1,group-argentina-10,Volunteers plant new trees in Argentina,Name the single country explicitly written in ...,Argentina,\nThe project is now in its third year.\n###Pr...,Argentina
2,group-australia-06,Volunteers in Australia plant new trees,"Copy the headline exactly, including punctuation.",Volunteers in Australia plant new trees,\nThe Australian government has been working w...,Volunteers in Australia plant new trees
3,group-australia-06,Volunteers in Australia plant new trees,Extract the one explicitly named country from ...,Australia,\nThe Australian government has been working w...,Australia
4,group-australia-07,A clean-up day in Australia draws neighbors,Repeat the input headline exactly as written.,A clean-up day in Australia draws neighbors,\nThe Australian government has announced that...,A clean-up day in Australia draws neighbors
5,group-australia-07,A clean-up day in Australia draws neighbors,Name the single country explicitly written in ...,Australia,\nThe Australian government has been criticise...,Australia


## Score the task contract

Use the raw decoded response after removing outer whitespace only. Do not lowercase, remove punctuation, normalize internal whitespace, or apply fuzzy matching: case and punctuation matter for exact copy.

The primary results are:

- **Per-task exact match:** separately count predictions equal to the reference for `exact_copy` and `extract_country`, and report numerator and denominator.
- **Paired both-correct rate:** for each test group, require both its copy and extraction responses to be exact; report correct groups out of 30.
- **Instruction-switch rate:** on each paired input, count whether the two generated responses are different when the two references are different. This is only a diagnostic: different responses can both be wrong, so read it beside exact match and paired correctness.

For the copy task only, report BLEU and chrF as secondary overlap context. BLEU compares word n-grams and includes a brevity penalty; chrF compares character n-grams, balancing character-level precision and recall. They can give partial credit to a near-copy, so they do not replace exact match. Country labels are often one or two words, making exact match the appropriate primary measure rather than a broad n-gram score.

Show counts as well as percentages. The metric cells report the values from the actual baseline and tuned generations, with no invented outcome.

In [9]:
table, scores = utils.scorecard(baseline_predictions, tuned_predictions)
display(table)

utils.save_evaluation(OUTPUT_DIR, baseline_predictions, tuned_predictions, scores)


,starting,tuned
Measure,,
Copy exact match,0/30 (0%),30/30 (100%)
Country exact match,0/30 (0%),30/30 (100%)
Both correct per headline,0/30 (0%),30/30 (100%)
Different responses (diagnostic),29/30 (97%),30/30 (100%)
Copy BLEU (overlap),0.17,100.00
Copy chrF (overlap),12.44,100.00


PosixPath('artifacts/full/evaluation.json')

## Interpret what the run shows

Begin with the paired generation table, then read the metrics and losses. Ask whether the tuned model produces the whole input for the copy instruction, only the named country for the extraction instruction, and different responses when only the instruction changes. A lower validation loss is encouraging for next-token prediction, but it is not accuracy and does not by itself prove that either task contract is reliable.

A high copy score with a low extraction score can mean that the model echoes well but does not use the instruction reliably. A high switch rate with low exact match means it changes behavior without reaching the requested answer. A good per-task average with a weaker paired both-correct rate means the two skills are not succeeding together on the same groups. Use the actual rows to support whichever pattern the run shows.

Keep the conclusion narrow. These prepared examples use familiar countries and a small family of headline forms, and the test set has only 30 groups. Success means that this controlled English copy/extract behavior changed under supervised fine-tuning; it does not establish broad instruction following, human-quality judgment, or performance on unrelated inputs.